# NPC EXPERIMENTAL BRAIN 

In [1]:
import numpy as np
from openai import OpenAI
from pydantic import BaseModel
from enum import Enum

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

LMSTUDIO_BASE_URL = os.environ["LMSTUDIO_BASE_URL"]
LM_API_TOKEN = os.environ["LM_API_TOKEN"]

# Identifiant EXACT du modele charge dans LM Studio (onglet Developer).
# Pour lister les ids disponibles : execute la cellule "models" plus bas.
MODEL = "google/gemma-4-e4b"

In [2]:
# client = ollama.Client()
client = OpenAI(base_url=LMSTUDIO_BASE_URL, api_key=LM_API_TOKEN)

NameError: name 'LMSTUDIO_BASE_URL' is not defined

In [ ]:
models = [m.id for m in client.models.list().data]
# models

# Couche de contrat

Contrat en anglais (UP / DOWN / LEFT / RIGHT). Les clés de perception et le prompt sont aussi en anglais (voir la partie Analyses en fin de notebook).

In [ ]:
# Constantes

VOID        = 0
PLAYER      = 1
ENEMY       = 2
GOLD        = 3
OBSTACLE    = 4
NPC         = 5

# Affichage de la carte dans le notebook (jamais fourni au LLM)
SYMBOLS = {VOID: "·", PLAYER: "👤", ENEMY: "👹", GOLD: "💰", OBSTACLE: "█"}

# Labels sémantiques fournis au LLM (perception 100 % anglais)
NAMES = {VOID: "EMPTY", PLAYER: "PLAYER", ENEMY: "ENEMY", GOLD: "GOLD", OBSTACLE: "OBSTACLE", NPC: "NPC"}

In [ ]:
class Direction(str, Enum):
    UP = "UP"
    DOWN = "DOWN"
    LEFT = "LEFT"
    RIGHT = "RIGHT"

class PlayerDecision(BaseModel):
    direction: Direction
    # decisionDetails: str

MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}

OPPOSITE = {"UP": "DOWN", "DOWN": "UP", "LEFT": "RIGHT", "RIGHT": "LEFT"}

# Moteur de perception

In [ ]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [ ]:
def compute_distance(entities_positions, pos_reference):
    if len(entities_positions) == 0:
        return np.array([])
    
    v = entities_positions - pos_reference
    distances = np.linalg.norm(v, axis=1)

    return np.round(distances, 2) # /!\ approx. distance

In [ ]:
def adjacent_cells(world_map, player_position):
    rows, cols = world_map.shape
    r, c = player_position
    adjacent = {}
    for name, (d_row, d_column) in MOVES.items():
        new_row, new_column = r + d_row, c + d_column
        if not (0 <= new_row < rows and 0 <= new_column < cols):
            adjacent[name] = "WALL"
        else:
            adjacent[name] = NAMES.get(world_map[new_row, new_column], "?")
    return adjacent

In [ ]:
def to_direction(delta_row, delta_column):
    dirs = []
    if delta_row < 0: dirs.append("UP")
    if delta_row > 0: dirs.append("DOWN")
    if delta_column < 0: dirs.append("LEFT")
    if delta_column > 0: dirs.append("RIGHT")
    return dirs

In [ ]:
def perception(world_map):
    player_position = localize(world_map, PLAYER)[0]
    gold_positions = localize(world_map, GOLD)
    enemy_positions = localize(world_map, ENEMY)

    gold_dist = compute_distance(gold_positions, player_position)
    enemy_dist = compute_distance(enemy_positions, player_position)

    closest_gold_direction = (
        to_direction(*(gold_positions[np.argmin(gold_dist)] - player_position))
        if len(gold_positions) > 0 else []
    )

    return {
        "enemies_within_radius_3": int(np.sum(enemy_dist <= 3)) if len(enemy_dist) > 0 else 0,
        "enemy_distances": enemy_dist.tolist(),
        "enemy_count": len(enemy_positions),
        "closest_enemy_distance": float(np.min(enemy_dist)) if len(enemy_dist) > 0 else 999,
        "gold_distances": gold_dist.tolist(),
        "closest_gold_distance": float(np.min(gold_dist)) if len(gold_dist) > 0 else 999,
        "closest_gold_direction": closest_gold_direction,
        "adjacent_cells": adjacent_cells(world_map, player_position),
    }

In [ ]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))

# Moteur de déplacement

In [ ]:
# Raisons d'échec d'un déplacement (fournies telles quelles au LLM)
BLOCK_MAP_EDGE = "MAP_EDGE"
BLOCK_ENEMY    = "ENEMY"
BLOCK_OBSTACLE = "OBSTACLE"

def blocked_reason(world_map: np.ndarray, pos):
    """Retourne None si la case est praticable, sinon la raison du blocage."""
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return BLOCK_MAP_EDGE

    if world_map[r, c] == ENEMY:
        return BLOCK_ENEMY

    if world_map[r, c] in (VOID, GOLD):
        return None

    return BLOCK_OBSTACLE

def allowed_move(world_map: np.ndarray, pos):
    return blocked_reason(world_map, pos) is None

In [ ]:
def move(world_map: np.ndarray, old_pos, new_pos):
    move_result = {
        "gold_collected": False,
        "new_pos": old_pos,
        "success": False,
        "failure_reason": None,
    }

    reason = blocked_reason(world_map, new_pos)
    if reason is not None:
        move_result["failure_reason"] = reason
        return move_result

    entity = world_map[old_pos[0], old_pos[1]]
    target = world_map[new_pos[0], new_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    move_result["new_pos"] = new_pos
    move_result["success"] = True

    if target == GOLD:
        move_result["gold_collected"] = True

    return move_result

# Charge algorithmique en plus

- `valid_directions` (bonus) : enlève les directions interdites (bord, ennemi, obstacle) avant l'appel au LLM. Activé avec `prefilter_directions=True`.
- `direction_analysis` (amélioration) : pour chaque direction possible, distances vers l'or et l'ennemi après le déplacement. L'algo informe, le LLM décide.

In [ ]:
def valid_directions(world_map: np.ndarray, player_position):
    """Bonus : filtre en amont les directions invalides (bord, ennemi, obstacle)
    avant de solliciter le LLM. Activable via prefilter_directions=True."""
    r, c = player_position
    return [
        name for name, (d_row, d_col) in MOVES.items()
        if allowed_move(world_map, (r + d_row, c + d_col))
    ]

In [ ]:
def direction_analysis(world_map: np.ndarray, player_position):
    """Amélioration : lookahead à 1 coup.

    Pour chaque direction praticable, distances résultantes vers l'or et
    l'ennemi les plus proches. L'algorithme informe, le LLM décide."""
    gold_positions = localize(world_map, GOLD)
    enemy_positions = localize(world_map, ENEMY)
    r, c = player_position

    analysis = {}
    for name, (d_row, d_col) in MOVES.items():
        new_pos = (r + d_row, c + d_col)
        if not allowed_move(world_map, new_pos):
            continue
        gold_dist = compute_distance(gold_positions, np.array(new_pos))
        enemy_dist = compute_distance(enemy_positions, np.array(new_pos))
        analysis[name] = {
            "closest_gold_distance": float(np.min(gold_dist)) if len(gold_dist) > 0 else 999,
            "closest_enemy_distance": float(np.min(enemy_dist)) if len(enemy_dist) > 0 else 999,
        }
    return analysis

# Moteur de décision

In [ ]:
def decide(player_perception: dict) -> PlayerDecision | None:
    prompt = f"""
    # Context
    - I am a player on a grid map and I want to collect gold.

    # Objective
    - Give me the single move that follows the shortest path toward the closest gold.

    # Rules
    - You must avoid enemies: never choose a direction whose adjacent cell contains ENEMY.
    - If "allowed_directions" is provided, you must pick a direction from this list only.
    - If "last_move_failed" is not null, the previous move failed for the given reason: do not repeat it.
    - Use "move_history" to detect and break oscillations: do not endlessly go back and forth.
    - If "direction_analysis" is provided, it gives for each walkable direction the resulting distances after that move: prefer moves that reduce the gold distance while keeping the enemy at a safe distance.

    # Player perception
    {player_perception} """

    # print(prompt)
    print(player_perception)

    response = client.beta.chat.completions.parse(
            model    = MODEL,
            messages = [{"role": "user", "content": prompt}],
            temperature=0,
            response_format=PlayerDecision
        )

    return response.choices[0].message.parsed or None

# Game loop (simulation)

In [ ]:
initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 0, 3],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3],
    [0, 0, 0, 0, 0, 0, 0], 
])

In [ ]:
HISTORY_SIZE = 10  # nb max d'entrées d'historique fournies au LLM

def game_loop(world_map: np.ndarray, max_turns = 10,
              use_history = True, use_lookahead = True, prefilter_directions = False):
    world_map = world_map.copy()

    move_history = []          # historique des déplacements (fourni au LLM)
    last_move_failure = None   # signalement de l'échec du dernier déplacement

    for turn in range(max_turns):
        print(f"\n =================== [Turn {turn + 1}] ===================")

        show_map(world_map)

        player_pos = localize(world_map, PLAYER)[0]

        p = perception(world_map)
        p["last_move_failed"] = last_move_failure
        if use_history:
            p["move_history"] = move_history[-HISTORY_SIZE:]
        if use_lookahead:
            p["direction_analysis"] = direction_analysis(world_map, player_pos)
        if prefilter_directions:
            p["allowed_directions"] = valid_directions(world_map, player_pos)

        decision: PlayerDecision | None = decide(p)

        if decision is None:
            print("\t → No decision returned by the LLM")
            continue

        direction = decision.direction.value
        print(f"\t → LLM decision: {direction}")

        d_row, d_col = MOVES[direction]
        new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)
        move_result = move(world_map, player_pos, new_pos)

        move_history.append({
            "turn": turn + 1,
            "direction": direction,
            "success": move_result["success"],
        })

        if move_result["success"]:
            last_move_failure = None
        else:
            last_move_failure = {
                "direction": direction,
                "reason": move_result["failure_reason"],
            }
            print(f"\t → Move failed: {move_result['failure_reason']}")

        if move_result["gold_collected"]:
            print("\n >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> FOUND GOLD ! <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< \n")
            break

In [ ]:
# Run principal : anglais + historique + signalement des échecs + lookahead
game_loop(world_map=initial_map, max_turns=10)

## Expérience A/B : impact de l'historique des déplacements

Deux runs comparatifs (sans lookahead ni filtrage pour isoler la variable) : d'abord **sans** historique, puis **avec** (cf. *Analyses*).

In [ ]:
# Sans historique (comportement de référence, sujet aux oscillations)
game_loop(world_map=initial_map, max_turns=10, use_history=False, use_lookahead=False)

In [ ]:
# Avec historique (le LLM peut détecter et casser les oscillations)
game_loop(world_map=initial_map, max_turns=10, use_history=True, use_lookahead=False)

## Bonus : filtrage actif des directions invalides en amont de l'appel LLM

In [ ]:
# Bonus : le LLM ne choisit plus que parmi les directions valides
game_loop(world_map=initial_map, max_turns=10, prefilter_directions=True)

# Analyses

## 1. Suppression des biais liés à la langue

J'ai mis en anglais les directions UP DOWN LEFT RIGHT les clés de la perception et le prompt. Les cases autour du joueur sont écrites avec des mots maintenant EMPTY ENEMY WALL GOLD au lieu des emojis.

Ce que je vois c'est qu'à modèle égal la trajectoire change pas trop sur cette petite carte. Le gain c'est surtout la cohérence parce qu'avant le prompt était en français alors que les clés étaient déjà en anglais. Et UP DOWN LEFT RIGHT c'est des mots que le modèle a vu beaucoup plus souvent que HAUT BAS GAUCHE DROITE. En gros ça enlève du bruit mais ça change pas la stratégie.

## 2. Historique des déplacements

J'ajoute move_history dans la perception donc le tour la direction et si le coup a marché. Je mets aussi une consigne dans le prompt pour pas tourner en rond.

Ce que je vois quand je compare le run sans historique et le run avec :

Sans historique le modèle se bloque il fait HAUT BAS HAUT BAS devant l'ennemi et il trouve jamais l'or sur les 10 tours.

Avec l'historique seul il arrête de répéter exactement les deux mêmes coups il teste aussi LEFT et RIGHT donc ça casse la boucle. Par contre avec le petit modèle 4B ça suffit pas il part un peu dans tous les sens et il atteint quand même pas l'or en 10 tours.

C'est vraiment quand je mets historique + lookahead ensemble (partie 4) qu'il atteint l'or en 7 tours.

Donc l'historique ça aide à pas rester bloqué bêtement mais tout seul ça résout pas la carte ici.

## 3. Perception des échecs de déplacement

Maintenant move() renvoie success et failure_reason qui peut être MAP_EDGE ENEMY ou OBSTACLE. La boucle garde le dernier échec dans last_move_failed et le remet dans la perception au tour d'après. Dès qu'un coup passe je remets à None.

Ça reste bien de la charge algo comme dans le sujet parce que c'est l'algo qui sait pourquoi le coup a raté donc il informe. Mais c'est le LLM qui choisit quoi faire avec grâce à la règle do not repeat it dans le prompt.

## 4. Amélioration lookahead à 1 coup (direction_analysis)

Pour chaque direction possible l'algo calcule d'avance les distances vers l'or et vers l'ennemi si le joueur y allait. C'est ça qui fait vraiment marcher la simu le modèle atteint l'or en 7 tours au lieu de tourner en rond.

Niveau traction le calcul de distance après le coup passe côté algo là où le LLM se plante souvent. Le LLM lui garde le choix genre est-ce que je vais vers l'or ou est-ce que je fuis l'ennemi. Du coup l'algo informe mieux mais il décide rien donc l'équilibre reste ok.

## Bonus filtrage des directions invalides (prefilter_directions=True)

valid_directions enlève les directions interdites (bord ennemi obstacle) avant même d'appeler le LLM et le prompt lui dit de choisir dans allowed_directions.

Là ce qui change c'est qu'au lieu de juste signaler l'échec après coup l'algo empêche carrément le mauvais coup avant. Donc une partie de la décision passe côté algo le LLM peut plus jouer un coup invalide. Dans le run bonus il atteint l'or en 7 tours sans aucun coup raté. On gagne en fiabilité mais du coup le LLM sert moins pour la sécurité donc la traction penche plus vers l'algo qu'avec juste le signalement.